# Tokenizer Analysis

This notebook demonstrates the custom BPE, byte-level BPE, WordPiece, and Unigram tokenizer implementations.

In [ ]:
from tokenizer import BPETokenizer, ByteLevelBPETokenizer, WordPieceTokenizer, UnigramTokenizer
from tokenizer.benchmark import benchmark_tokenizers, format_benchmark_results

corpus = [
    'Tokenization converts text into model friendly token ids.',
    'Byte pair encoding learns frequent subword patterns.',
    'Unicode examples include Ελληνικά, café, and 🚀.',
]
sample = 'Tokenization handles Ελληνικά and 🚀.'

## Train all tokenizers

In [ ]:
tokenizers = {
    'bpe': BPETokenizer(vocab_size=120, min_frequency=1),
    'byte_bpe': ByteLevelBPETokenizer(vocab_size=330, min_frequency=1),
    'wordpiece': WordPieceTokenizer(vocab_size=120, min_frequency=1),
    'unigram': UnigramTokenizer(vocab_size=120, min_frequency=1),
}
for tokenizer in tokenizers.values():
    tokenizer.fit(corpus)

{name: len(tok.get_vocab()) for name, tok in tokenizers.items()}

## Compare token lengths and decoded text

In [ ]:
for name, tokenizer in tokenizers.items():
    ids = tokenizer.encode(sample)
    print(name, len(ids), ids)
    print('decoded:', tokenizer.decode(ids))
    print()

## Inspect BPE merge rules

In [ ]:
tokenizers['bpe'].merges[:10], tokenizers['byte_bpe'].merges[:10]

## Trace merge operations

In [ ]:
tokenizers['bpe'].trace_encode('Tokenization')

## Benchmark custom tokenizers

In [ ]:
results = benchmark_tokenizers(tokenizers, corpus, benchmark_runs=3, warmup_runs=1)
print(format_benchmark_results(results))

## Tradeoffs

- BPE is compact and easy to inspect, but character initialization is less robust for arbitrary Unicode.
- Byte-level BPE can represent any UTF-8 input, often at the cost of longer initial sequences.
- WordPiece gives predictable continuation markers and greedy segmentation.
- Unigram uses a probabilistic scoring view and Viterbi segmentation, but this simplified trainer does not implement EM pruning.